# 🕉️ BhajanForge — Voice Cloning (clean, reset-resilient)

Trains an **RVC** model of **your own voice** on a free Colab T4 GPU, saves it to
your **Google Drive** (so a runtime reset can't lose it), and downloads it.

### Run order (top to bottom)
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run Step 0 → Step 5 in order. Read the note above each cell.
3. Keep this tab **open and active** — free runtimes recycle when idle.

> ⚠️ Only clone **your own** voice (you are the artist).


## Step 0 — GPU check
Must say `GPU OK`. If not: Runtime → Change runtime type → T4 GPU, then re-run.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then re-run.'
print('GPU OK:', torch.cuda.get_device_name(0))

## Step 1 — Install (one cell, fixed)
Clones RVC and installs everything. Uses a build-patched `fairseq` fork and
installs each dependency individually so one failure can't block the rest.
Takes a few minutes; prints `OK`/`FAIL` per package, then `install OK`.

In [ ]:
%cd /content
import sys; print('Python', sys.version.split()[0])

!pip -q install yt-dlp demucs soundfile librosa

![ -d RVC ] || git clone -q https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI RVC
%cd /content/RVC

!pip -q install "setuptools<82" wheel Cython
!pip -q install --no-build-isolation "git+https://github.com/One-sixth/fairseq.git"

for d in ["faiss-cpu","praat-parselmouth","pyworld","torchcrepe","librosa",
          "scipy","soundfile","ffmpeg-python","tqdm","einops","tensorboardX",
          "matplotlib","audioread","av","numba","resampy","pydub"]:
    !pip -q install "{d}" || print("WARN could not install", d)

import importlib
ok = True
for m in ['torch','fairseq','faiss','parselmouth','pyworld','torchcrepe','librosa','scipy']:
    try:
        importlib.import_module(m); print('OK   ', m)
    except Exception as e:
        ok = False; print('FAIL ', m, '->', repr(e))
print('install OK' if ok else 'INSTALL HAD FAILURES - paste the FAIL lines to BhajanForge')

## Step 1b — Download RVC pretrained assets
HuBERT + rmvpe pitch model + pretrained generators (needed for training).

In [ ]:
%cd /content/RVC
import os
for sub in ['assets/hubert','assets/rmvpe','assets/pretrained_v2']:
    os.makedirs(sub, exist_ok=True)
base = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'
!test -f assets/hubert/hubert_base.pt || wget -q {base}/hubert_base.pt -O assets/hubert/hubert_base.pt
!test -f assets/rmvpe/rmvpe.pt || wget -q {base}/rmvpe.pt -O assets/rmvpe/rmvpe.pt
!test -f assets/pretrained_v2/f0G40k.pth || wget -q {base}/pretrained_v2/f0G40k.pth -O assets/pretrained_v2/f0G40k.pth
!test -f assets/pretrained_v2/f0D40k.pth || wget -q {base}/pretrained_v2/f0D40k.pth -O assets/pretrained_v2/f0D40k.pth
print('assets ready:', os.listdir('assets/pretrained_v2'))

## Step 2 — Add YOUR voice clips
Click the **folder icon** (left sidebar) and **drag your `.mp3`/`.wav` files into
it** (they land in `/content`). This cell moves them into `/content/raw`.
*(Or paste your own YouTube links in the list.)*

In [ ]:
YOUTUBE_URLS = [
    # 'https://www.youtube.com/watch?v=XXXXXXXXXXX',
]
import os, glob, shutil
os.makedirs('/content/raw', exist_ok=True)
for f in (glob.glob('/content/*.mp3') + glob.glob('/content/*.wav')
          + glob.glob('/content/*.m4a') + glob.glob('/content/*.flac')):
    shutil.move(f, '/content/raw/' + os.path.basename(f))
for url in YOUTUBE_URLS:
    !yt-dlp -x --audio-format wav -o '/content/raw/%(id)s.%(ext)s' "{url}"
print('raw files:', os.listdir('/content/raw'))
assert os.listdir('/content/raw'), 'No audio yet - upload files (file panel) or add YouTube URLs.'

## Step 3 — Isolate clean vocals
Demucs strips the music, keeping only your voice. Output goes to `/content/dataset`.

In [ ]:
import os, glob, shutil
os.makedirs('/content/dataset', exist_ok=True)
for f in glob.glob('/content/raw/*'):
    !demucs --two-stems=vocals -o /content/sep "{f}"
n = 0
for v in glob.glob('/content/sep/*/*/vocals.wav'):
    shutil.copy(v, f'/content/dataset/vocal_{n}.wav'); n += 1
print('vocal clips for training:', n)
assert n > 0, 'No vocals isolated - add audio in Step 2 and re-run.'

## Step 4 — Train your voice model
Set `MODEL_NAME` and `EPOCHS` (150 trains faster = less chance of a reset; raise
to 200-300 later for higher quality). Runs preprocess → feature/f0 → index → train.

In [ ]:
MODEL_NAME = 'shyam_voice_v1'
SAMPLE_RATE = 40000
EPOCHS = 150
import os
%cd /content/RVC
os.environ['MODEL_NAME'] = MODEL_NAME

# 1) preprocess
!python infer/modules/train/preprocess.py /content/dataset {SAMPLE_RATE} 2 /content/RVC/logs/{MODEL_NAME} False 3.0
# 2) extract pitch (rmvpe) + HuBERT features
!python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 /content/RVC/logs/{MODEL_NAME} True
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 /content/RVC/logs/{MODEL_NAME} v2 True
print('preprocess + feature extraction done')

In [ ]:
# 3) build the faiss retrieval index
import numpy as np, faiss, glob
feat_dir = f'/content/RVC/logs/{MODEL_NAME}/3_feature768'
feats = [np.load(f) for f in sorted(glob.glob(feat_dir + '/*.npy'))]
big = np.concatenate(feats, 0).astype('float32')
index = faiss.index_factory(768, 'IVF{},Flat'.format(max(1, int(np.sqrt(len(big))))))
index.train(big); index.add(big)
idx_path = f'/content/RVC/logs/{MODEL_NAME}/added_{MODEL_NAME}.index'
faiss.write_index(index, idx_path)
print('index written:', idx_path)

In [ ]:
# 4) train (writes weights/<MODEL_NAME>.pth). ~10-25 min on T4 at 150 epochs.
%cd /content/RVC
!python infer/modules/train/train.py -e {MODEL_NAME} -sr 40k -f0 1 -bs 8 -te {EPOCHS} -se 50 \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth \
  -l 0 -c 0 -sw 1 -v v2
import glob
print('trained weights:', glob.glob('/content/RVC/assets/weights/*.pth') + glob.glob(f'/content/RVC/logs/{MODEL_NAME}/*.pth'))

## Step 5 — Save to Google Drive + download
Copies your `.pth` + `.index` to Google Drive (survives runtime resets) **and**
downloads a zip to your PC.

In [ ]:
import glob, os, shutil
MODEL_NAME = MODEL_NAME if 'MODEL_NAME' in globals() else 'shyam_voice_v1'
weights = glob.glob('/content/RVC/assets/weights/*.pth') or glob.glob(f'/content/RVC/logs/{MODEL_NAME}/*.pth')
assert weights, 'No trained .pth found - run Step 4 first.'
pth = sorted(weights)[-1]
index = (glob.glob(f'/content/RVC/logs/{MODEL_NAME}/added_*.index') + [''])[0]

os.makedirs('/content/model_out', exist_ok=True)
shutil.copy(pth, f'/content/model_out/{MODEL_NAME}.pth')
if index:
    shutil.copy(index, f'/content/model_out/added_{MODEL_NAME}.index')

# save to Drive (resilient)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/bhajan_voice'
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob('/content/model_out/*'):
        shutil.copy(f, dst)
    print('Saved to Google Drive:', dst, os.listdir(dst))
except Exception as e:
    print('Drive save skipped:', e)

# download zip to PC
shutil.make_archive('/content/' + MODEL_NAME + '_model', 'zip', '/content/model_out')
from google.colab import files
files.download('/content/' + MODEL_NAME + '_model.zip')
print('Downloaded', MODEL_NAME + '_model.zip - now do Step 6 (host on Hugging Face).')

## Step 6 — Host on Hugging Face + connect BhajanForge
1. Unzip the downloaded `*_model.zip` (or grab it from Google Drive → `bhajan_voice`).
2. In BhajanForge open **`hf_space/README.md`**: create a free Hugging Face
   **Docker Space**, upload `Dockerfile`, `app.py`, `requirements.txt`, and drop
   your `.pth` + `.index` into the Space's **`models/`** folder.
3. When the Space shows **Running**, set in BhajanForge `.env`:
   ```
   VOICE_PROVIDER=colab_tunnel
   RVC_TUNNEL_URL=https://<your-username>-<space>.hf.space
   ```
   and `config/learning.yaml` → `voice_profile.active_rvc_model: shyam_voice_v1`.
4. Run a `produce`. Your bhajan comes back in **your** cloned voice.
